# Lab 4: Network Routing — Shortest Paths on the Christchurch Road Network
## ENCN205 — Applied Data Analysis for Civil and Natural Systems
### University of Canterbury | Te Whare Wānanga o Waitaha

**Duration:** 2–3 hours
**Week:** 4 (GIS labs)
**Prerequisites:** Labs 1–3 completed (GeoPandas basics, CRS, attribute and spatial queries, `.explore()` maps)

---

### Learning Objectives
By the end of this lab you will be able to:
1. Explain how a road network is represented as a **graph** of nodes, edges and weights
2. Build a graph (a Python dictionary) from a GeoDataFrame of road segments
3. Implement **Dijkstra's shortest-path algorithm** from pseudocode and test it
4. Snap real-world locations to the nearest network node and route between them
5. Compare the **shortest** (distance) and **fastest** (time) routes
6. Model a disruption — bridge closures — and quantify the detour it causes
7. Compute and map a travel-time **service area** (isochrone)

### Engineering Scenario
> **You work for the emergency management team at Christchurch City Council.**
> On 22 February 2011 several bridges over the Avon River / Ōtākaro in the eastern
> suburbs — Bridge Street, Pages Road, Anzac Drive, Gayhurst Road, Avondale Road — were
> damaged and closed, cutting the usual routes between New Brighton and the central city.
>
> Your task is to build a **routing tool** — the engine behind every navigation app — that
> finds the fastest route between two places on the real Christchurch road network, and
> then use it to measure how much the bridge closures lengthened the trip from the eastern
> suburbs to Christchurch Hospital.
>
> The network comes from OpenStreetMap: about 970 km of road, 8,659 segments and
> 6,946 intersections covering the central city and the eastern suburbs.

**Lecture links:** connectivity and network distance (L5), CRS and metric lengths (L4),
map design (L7), critical infrastructure and cascading failures (L10).

## 1. Setup and Loading the Road Network

The data is a GeoPackage, `christchurch_roads.gpkg`, with four layers (all in **NZTM2000,
EPSG:2193**, so lengths are in metres):

| Layer | Geometry | What it holds |
|-------|----------|---------------|
| `edges` | LineString | 8,659 road segments between intersections. Key columns: `u`, `v` (the node at each end), `name`, `highway` (road class), `speed_kmh`, `length_m`, `travel_s`, `is_bridge`, `bridge_name`, `river` (name of the river a bridge spans) |
| `nodes` | Point | 6,946 intersections and dead ends. `node_id`, `degree` (number of connected segments) |
| `places` | Point | 23 origins and destinations: hospitals, fire and police stations, marae, landmarks |
| `rivers` | LineString | Avon / Ōtākaro and Heathcote / Ōpāwaho centrelines, for context |

> **Where does it come from?** The segments were downloaded from OpenStreetMap. Two segments
> are connected only if they share a node in OpenStreetMap — that is how an overpass can cross
> a motorway without joining it. `travel_s` is simply `length_m / speed` with the posted speed
> limit, so it is a free-flow estimate with no traffic and no time lost at intersections.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

In [ ]:
# Load the four layers
gpkg = 'data/christchurch_roads.gpkg'
edges  = gpd.read_file(gpkg, layer='edges')
nodes  = gpd.read_file(gpkg, layer='nodes')
places = gpd.read_file(gpkg, layer='places')
rivers = gpd.read_file(gpkg, layer='rivers')

print(f"edges:  {len(edges):>5} segments, CRS {edges.crs.to_epsg()}")
print(f"nodes:  {len(nodes):>5} intersections")
print(f"places: {len(places):>5} origins / destinations")
print(f"\nTotal road length: {edges['length_m'].sum() / 1000:.0f} km")
print("\nRoad classes:")
print(edges['highway'].value_counts().to_string())
edges.head()

In [ ]:
# The places we can route between
places[['name', 'category']].sort_values('category').reset_index(drop=True)

In [ ]:
# Overview map: the network, the two rivers, and every bridge that spans a river
fig, ax = plt.subplots(figsize=(12, 10))
edges.plot(ax=ax, color='#b0b8bd', linewidth=0.5, zorder=1)
edges[edges['highway'].isin(['motorway', 'trunk', 'primary'])].plot(
    ax=ax, color='#5a6b74', linewidth=1.2, zorder=2, label='Main roads')
rivers.plot(ax=ax, color='#2980b9', linewidth=2.5, zorder=3, label='Rivers')
edges[edges['river'].notna()].plot(ax=ax, color='#c0392b', linewidth=4, zorder=4,
                                   label='Road bridges over rivers')
places.plot(ax=ax, column='category', legend=True, markersize=60, zorder=5,
            edgecolor='black', linewidth=0.5, cmap='Set1')
for _, p in places[places['category'] == 'hospital'].iterrows():
    ax.annotate(p['name'], (p.geometry.x, p.geometry.y), xytext=(6, 6),
                textcoords='offset points', fontsize=9, fontweight='bold')
ax.set_title('Christchurch road network — central city and eastern suburbs (OpenStreetMap)', fontsize=13)
ax.set_xlabel('Easting (m, NZTM2000)')
ax.set_ylabel('Northing (m, NZTM2000)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 2. From Roads to a Graph

A navigation app does not search over line geometries. It searches over a **graph**:

| Graph term | On a road network | In our data |
|------------|-------------------|-------------|
| **Node** (vertex) | an intersection or dead end | one row of `nodes`, identified by `node_id` |
| **Edge** (arc) | a road segment joining two intersections | one row of `edges`, joining `u` and `v` |
| **Weight** (cost) | how much it costs to travel the edge | `length_m` (distance) or `travel_s` (time) |

This is the *connectivity* relationship from Lecture 5: what matters is not where segments
are, but which ones are **joined**. Two roads that cross on the map are only connected if they
share a node — an overpass shares none.

### The adjacency dictionary
The simplest way to store a graph in Python is a *dictionary of dictionaries*:

```python
graph[u][v] = weight     # cost of travelling from node u to its neighbour v
```

so `graph[100]` is a small dictionary listing every neighbour of node 100 and the cost of
reaching it. We treat every road as two-way (a simplification: real one-way streets are
ignored), so each edge appears twice: `graph[u][v]` and `graph[v][u]`.

In [ ]:
def build_graph(edges, weight_column):
    """Adjacency dictionary from an edges GeoDataFrame: graph[u][v] = weight of edge u-v."""
    graph = {}
    for u, v, w in zip(edges['u'], edges['v'], edges[weight_column]):
        graph.setdefault(u, {})
        graph.setdefault(v, {})
        # if two segments join the same pair of nodes, keep the cheaper one
        graph[u][v] = min(w, graph[u].get(v, float('inf')))
        graph[v][u] = graph[u][v]
    return graph

# A graph weighted by DISTANCE (metres)
dist_graph = build_graph(edges, 'length_m')

print(f"{len(dist_graph)} nodes in the graph")
node = 100
print(f"\nNode {node} has {len(dist_graph[node])} neighbours:")
for neighbour, metres in dist_graph[node].items():
    print(f"   -> node {neighbour}: {metres:.0f} m")

In [ ]:
# How connected are the intersections? (degree = number of segments meeting at a node)
degree_counts = nodes['degree'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
degree_counts.plot.bar(ax=ax, color='#0f6b5e', edgecolor='white')
ax.set_xlabel('Degree (segments meeting at the node)')
ax.set_ylabel('Number of nodes')
ax.set_title('Dead ends (1), T-junctions (3) and crossroads (4) dominate')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

### 🔧 YOUR TURN: A Time-Weighted Graph

Distance is not what drivers minimise — time is. Build a second graph weighted by `travel_s`
(seconds at the speed limit), then compare node 100's neighbours in the two graphs.

In [ ]:
# 🔧 YOUR TURN — time-weighted graph. Fill in the ... blanks.

# TODO 1: Build the graph with travel_s as the weight (same call as dist_graph above).
time_graph = ...

# TODO 2: Print node 100's neighbours with metres AND seconds side by side.
#   Pattern: loop over dist_graph[node].items() and look up time_graph[node][neighbour]
node = 100
for neighbour, metres in dist_graph[node].items():
    seconds = ...
    print(f"node {node} -> {neighbour}: {metres:6.0f} m   {seconds:5.1f} s")

print(f"\nSame {len(time_graph)} nodes? {len(time_graph) == len(dist_graph)}")

## 3. Dijkstra's Algorithm

Edsger Dijkstra published his shortest-path algorithm in 1959 — he later said he designed it
in about twenty minutes in a café in Amsterdam. It is still the core of every routing engine.

### The idea
Start at the source with a *tentative* distance of 0 to itself and infinity to everything else.
Repeatedly pick the **unvisited node that is currently closest**, mark it as visited (its
distance is now final — no shorter route to it can exist, because every other unvisited node is
already further away), and check whether going *through* it gives any of its neighbours a
shorter tentative distance. This last step is called **relaxing** an edge.

### Pseudocode

```
DIJKSTRA(graph, source, target):

    # 1. Initialise
    for every node n in graph:
        distance[n] = infinity
        previous[n] = nothing
    distance[source] = 0
    unvisited = the set of all nodes

    # 2. Main loop
    while unvisited is not empty:
        current = the node in unvisited with the smallest distance
        if distance[current] is infinity:  stop      # what is left cannot be reached
        if current is the target:          stop      # the target's distance is final
        remove current from unvisited

        # 3. Relax every edge leaving current
        for each (neighbour, weight) of current:
            new_distance = distance[current] + weight
            if new_distance < distance[neighbour]:
                distance[neighbour] = new_distance
                previous[neighbour] = current

    return distance, previous
```

`previous` records, for every node, which node we came from. Walking `previous` backwards
from the target to the source gives the route.

### Worked example — a toy network
Before running on 6,946 intersections, do it by hand on six. Weights are travel minutes.

In [ ]:
# A toy network: six nodes, nine edges (weights = minutes)
toy_graph = {
    'A': {'B': 2, 'C': 4},
    'B': {'A': 2, 'C': 1, 'D': 4},
    'C': {'A': 4, 'B': 1, 'D': 2, 'E': 1},
    'D': {'B': 4, 'C': 2, 'E': 2, 'F': 3},
    'E': {'C': 1, 'D': 2, 'F': 3},
    'F': {'D': 3, 'E': 3},
}

# Draw it
pos = {'A': (0, 1), 'B': (1, 2), 'C': (1, 0), 'D': (2, 2), 'E': (2, 0), 'F': (3, 1)}
fig, ax = plt.subplots(figsize=(7, 4))
drawn = set()
for u, nbrs in toy_graph.items():
    for v, w in nbrs.items():
        if (v, u) in drawn:
            continue
        drawn.add((u, v))
        (x1, y1), (x2, y2) = pos[u], pos[v]
        ax.plot([x1, x2], [y1, y2], color='#5a6b74', linewidth=2, zorder=1)
        ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.08, str(w), ha='center', fontsize=11,
                color='#c0392b', fontweight='bold', zorder=3)
for n, (x, y) in pos.items():
    ax.scatter(x, y, s=700, color='#0f6b5e', zorder=2)
    ax.text(x, y, n, ha='center', va='center', color='white', fontsize=13, fontweight='bold', zorder=4)
ax.set_xlim(-0.4, 3.4); ax.set_ylim(-0.5, 2.5); ax.axis('off')
ax.set_title('Toy network — what is the quickest way from A to F?')
plt.show()

Trace the algorithm from **A** by hand. Each row is one pass through the main loop; the
*visited* node's distance becomes final (bold).

| Step | Visited | A | B | C | D | E | F | Relaxed edges |
|:----:|:-------:|:-:|:-:|:-:|:-:|:-:|:-:|---------------|
| 0 | — | **0** | ∞ | ∞ | ∞ | ∞ | ∞ | initialise |
| 1 | A | 0 | **2** | 4 | ∞ | ∞ | ∞ | A→B (2), A→C (4) |
| 2 | B | 0 | 2 | **3** | 6 | ∞ | ∞ | B→C (2+1=3 < 4 ✔), B→D (6) |
| 3 | C | 0 | 2 | 3 | **5** | 4 | ∞ | C→D (3+2=5 < 6 ✔), C→E (4) |
| 4 | E | 0 | 2 | 3 | 5 | 4 | **7** | E→D (6, not < 5 ✘), E→F (7) |
| 5 | D | 0 | 2 | 3 | 5 | 4 | 7 | D→F (5+3=8, not < 7 ✘) |
| 6 | F = target | stop | | | | | | |

**Shortest A → F is 7 minutes**, and following `previous` backwards, F ← E ← C ← B ← A,
gives the route **A, B, C, E, F**. Notice how the route via D looked promising (B→D) until
relaxation found a better way through C.

### 🔧 YOUR TURN: Implement Dijkstra

Complete the function below by translating the pseudocode **line by line**. The
initialisation and the two stopping rules are done; you write the selection step and the
relaxation step. Then run the test cell — it checks your function against the hand-worked
example. If an `assert` fails, add `print(current, distance)` inside the loop and compare
with the table above.

In [ ]:
# 🔧 YOUR TURN — Dijkstra's algorithm. Fill in the ... blanks by following the pseudocode.

def dijkstra(graph, source, target=None):
    """Shortest paths from source through graph (an adjacency dictionary).

    Returns two dictionaries: distance[node] = cost of the best route found,
    previous[node] = the node before it on that route. If target is given the
    search stops as soon as the target is settled; otherwise it settles every node.
    """
    # 1. Initialise
    distance = {node: float('inf') for node in graph}
    previous = {node: None for node in graph}
    distance[source] = 0
    unvisited = set(graph)

    # 2. Main loop
    while unvisited:
        # TODO 1: the unvisited node with the smallest tentative distance.
        #   Hint: min(unvisited, key=distance.get)
        current = ...

        if distance[current] == float('inf'):
            break                      # what is left cannot be reached
        if current == target:
            break                      # the target's distance is now final
        unvisited.remove(current)

        # 3. Relax every edge leaving current
        for neighbour, weight in graph[current].items():
            # TODO 2: distance to neighbour if we go through current
            new_distance = ...
            # TODO 3: if that is an improvement, record it and remember where we came from
            if ...:
                distance[neighbour] = ...
                previous[neighbour] = ...

    return distance, previous

In [ ]:
def reconstruct_path(previous, source, target):
    """Walk the previous-dictionary backwards from target to source. Returns a list of nodes."""
    if previous.get(target) is None and target != source:
        return []                       # target was never reached
    path = [target]
    while path[-1] != source:
        path.append(previous[path[-1]])
    return path[::-1]


# --- TEST against the hand-worked example. Do not edit: your dijkstra() must pass this. ---
distance, previous = dijkstra(toy_graph, 'A', 'F')
route = reconstruct_path(previous, 'A', 'F')
print("A -> F:", distance['F'], "minutes via", route)

assert distance['F'] == 7,                  "distance should be 7 — compare with the table"
assert route == ['A', 'B', 'C', 'E', 'F'],  "route should be A, B, C, E, F"
assert distance['D'] == 5,                  "D should be reached in 5 via C, not 6 via B"
print("\nAll tests passed — your Dijkstra works!")

## 4. Routing on the Real Network

### Snapping places to the network
A hospital or a marae is not a node of the graph, so first we find the **nearest node** to it
— the same `distance()` idea you used for proximity in Lab 2. Then we route node to node.

In [ ]:
def nearest_node(point):
    """node_id of the network node closest to a shapely Point (NZTM coordinates)."""
    return int(nodes.loc[nodes.distance(point).idxmin(), 'node_id'])

def place(name):
    """Geometry of a named place."""
    return places.loc[places['name'] == name, 'geometry'].iloc[0]

origin      = nearest_node(place('New Brighton Pier'))
destination = nearest_node(place('Christchurch Hospital'))
print(f"New Brighton Pier     -> node {origin}, "
      f"{nodes.loc[origin].geometry.distance(place('New Brighton Pier')):.0f} m away")
print(f"Christchurch Hospital -> node {destination}, "
      f"{nodes.loc[destination].geometry.distance(place('Christchurch Hospital')):.0f} m away")

### Fastest versus shortest
Run your Dijkstra twice — once on `time_graph`, once on `dist_graph`. The helpers below turn
a list of nodes back into the road segments it follows (so we can measure and map it) and into
a list of street names, like the turn-by-turn panel in a navigation app.

In [ ]:
# Look-up from a pair of nodes to the row of `edges` that joins them
edge_index = {}
for i, (u, v) in enumerate(zip(edges['u'], edges['v'])):
    edge_index.setdefault((u, v), i)
    edge_index.setdefault((v, u), i)

def route_edges(path):
    """GeoDataFrame of the edges along a node path, in travel order."""
    rows = [edge_index[(a, b)] for a, b in zip(path[:-1], path[1:])]
    return edges.iloc[rows]

def street_list(route):
    """Street names along a route, without consecutive repeats."""
    streets = []
    for name in route['name']:
        if isinstance(name, str) and (not streets or streets[-1] != name):
            streets.append(name)
    return streets

def summarise(route, label):
    km = route['length_m'].sum() / 1000
    minutes = route['travel_s'].sum() / 60
    print(f"{label:<28} {km:5.2f} km   {minutes:5.1f} min   {len(route)} segments")
    return km, minutes


# Fastest route (time) and shortest route (distance)
dist_t, prev_t = dijkstra(time_graph, origin, destination)
fastest  = route_edges(reconstruct_path(prev_t, origin, destination))

dist_d, prev_d = dijkstra(dist_graph, origin, destination)
shortest = route_edges(reconstruct_path(prev_d, origin, destination))

print("New Brighton Pier -> Christchurch Hospital\n")
summarise(fastest,  'Fastest route (time)')
summarise(shortest, 'Shortest route (distance)')
print("\nFastest route follows:", ' -> '.join(street_list(fastest)))

In [ ]:
# Map both routes
fig, ax = plt.subplots(figsize=(12, 9))
edges.plot(ax=ax, color='#d0d6d9', linewidth=0.5, zorder=1)
rivers.plot(ax=ax, color='#2980b9', linewidth=2, zorder=2)
shortest.plot(ax=ax, color='#e67e22', linewidth=4, zorder=3,
              label=f"Shortest: {shortest['length_m'].sum()/1000:.1f} km, {shortest['travel_s'].sum()/60:.1f} min")
fastest.plot(ax=ax, color='#0f6b5e', linewidth=3, zorder=4,
             label=f"Fastest:  {fastest['length_m'].sum()/1000:.1f} km, {fastest['travel_s'].sum()/60:.1f} min")
for name, marker in [('New Brighton Pier', '^'), ('Christchurch Hospital', 's')]:
    p = place(name)
    ax.scatter(p.x, p.y, s=180, marker=marker, color='black', zorder=5)
    ax.annotate(name, (p.x, p.y), xytext=(8, 8), textcoords='offset points', fontsize=10, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_title('Fastest vs shortest route — New Brighton Pier to Christchurch Hospital', fontsize=13)
ax.set_xlabel('Easting (m)'); ax.set_ylabel('Northing (m)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

### 🔧 YOUR TURN: Route from Sumner

Repeat the analysis for **Sumner Beach → Christchurch Hospital**: find the fastest and the
shortest route, print the summary line for each, and list the streets of the fastest route.
How many minutes does a driver lose by taking the shortest-distance route instead?

In [ ]:
# 🔧 YOUR TURN — Sumner Beach to Christchurch Hospital. Fill in the ... blanks.
# Every command here appeared in the New Brighton example above.

# TODO 1: Snap both places to the network.
origin_s      = ...
destination_s = ...

# TODO 2: Fastest route (time_graph) and shortest route (dist_graph).
dist_t, prev_t = ...
fastest_s = route_edges(reconstruct_path(prev_t, origin_s, destination_s))

dist_d, prev_d = ...
shortest_s = route_edges(reconstruct_path(prev_d, origin_s, destination_s))

print("Sumner Beach -> Christchurch Hospital\n")
km_f, min_f = summarise(fastest_s,  'Fastest route (time)')
km_s, min_s = summarise(shortest_s, 'Shortest route (distance)')

# TODO 3: Minutes lost by taking the shortest-distance route.
minutes_lost = ...
print(f"\nTaking the shortest route costs an extra {minutes_lost:.1f} minutes")
print("\nFastest route follows:", ' -> '.join(street_list(fastest_s)))

## 5. Interactive Map

Now make it look like a navigation app. `.explore()` (Lab 2) draws GeoDataFrames on a
zoomable web map — remember to convert to **WGS84 (EPSG:4326)** first, because web maps use
latitude and longitude. Hover over a segment to see its street name and speed limit.

In [ ]:
# Interactive map of the two New Brighton routes (web maps need WGS84)
from IPython.display import display
import folium

try:
    m = shortest.to_crs(epsg=4326).explore(
        color='#e67e22',
        style_kwds={'weight': 6, 'opacity': 0.8},
        tooltip=['name', 'highway', 'speed_kmh', 'length_m'],
        tiles='OpenStreetMap',
        name='Shortest route (distance)'
    )
    fastest.to_crs(epsg=4326).explore(
        m=m,
        color='#0f6b5e',
        style_kwds={'weight': 5, 'opacity': 0.9},
        tooltip=['name', 'highway', 'speed_kmh', 'length_m'],
        name='Fastest route (time)'
    )
    places.to_crs(epsg=4326).explore(
        m=m,
        column='category',
        cmap='Set1',
        tooltip=['name', 'category'],
        marker_kwds={'radius': 7},
        style_kwds={'fillOpacity': 0.9, 'color': 'black', 'weight': 1},
        legend=True,
        name='Places'
    )
    folium.LayerControl().add_to(m)
    display(m)
except ImportError:
    print("Install folium for interactive maps: pip install folium")

## 6. Disruption: The 2011 Bridge Closures

Lecture 10 described the Christchurch earthquake as a **cascading failure** — damage to one
system knocked out others. Bridges are classic critical links: close one and every route that
used it must divert. Our `edges` layer marks each bridge segment (`is_bridge`) and, when it spans
a river, names the river in the `river` column.

Modelling a closure is simple: **remove the closed edges and rebuild the graph.** Dijkstra does
the rest.

In [ ]:
# Which road bridges cross the Avon River in our study area?
avon_bridges = edges[edges['river'].str.contains('Avon', na=False)]
print(f"{len(avon_bridges)} bridge segments spanning the Avon, on these roads:")
print(sorted(avon_bridges['bridge_name'].dropna().unique()))

In [ ]:
# Close ONE bridge (Pages Road) and see what happens to the New Brighton -> hospital trip
closed = ['Pages Road']

open_edges = edges[~(edges['river'].notna() & edges['bridge_name'].isin(closed))]
print(f"Removed {len(edges) - len(open_edges)} bridge segments; {len(open_edges)} edges remain")

time_graph_closed = build_graph(open_edges, 'travel_s')

_, prev_closed = dijkstra(time_graph_closed, origin, destination)
detour = route_edges(reconstruct_path(prev_closed, origin, destination))

print("\nNew Brighton Pier -> Christchurch Hospital")
km0, min0 = summarise(fastest, 'All bridges open')
km1, min1 = summarise(detour,  'Pages Road bridge closed')
print(f"\nExtra travel time: {min1 - min0:.1f} minutes")
print("Detour follows:", ' -> '.join(street_list(detour)))

### 🔧 YOUR TURN: The February 2011 Scenario

On 22 February 2011 the eastern Avon bridges were all closed at once. Model the closure of the
six bridges listed below and reroute **New Brighton Police Station → Christchurch Hospital**.
Report the trip time before and after, the extra minutes, and the streets of the detour, then
map both routes.

In [ ]:
# 🔧 YOUR TURN — six eastern Avon bridges closed. Fill in the ... blanks.
# Every command here appeared in the Pages Road example above.

closed_2011 = ['Bridge Street', 'Pages Road', 'Anzac Drive', 'Wainoni Road',
               'Avondale Road', 'Gayhurst Road']

# TODO 1: Keep only the edges that are NOT (a river bridge AND in the closed list).
open_2011 = ...
print(f"Removed {len(edges) - len(open_2011)} bridge segments")

# TODO 2: Rebuild the time-weighted graph from the remaining edges.
graph_2011 = ...

# TODO 3: Snap the police station and the hospital, then route on BOTH graphs.
origin_p      = ...
destination_p = ...

_, prev_open = ...
route_open = route_edges(reconstruct_path(prev_open, origin_p, destination_p))

_, prev_2011 = ...
route_2011 = route_edges(reconstruct_path(prev_2011, origin_p, destination_p))

print("\nNew Brighton Police Station -> Christchurch Hospital")
km_o, min_o = summarise(route_open, 'All bridges open')
km_c, min_c = summarise(route_2011, 'Six bridges closed (2011)')
print(f"\nExtra travel time: {min_c - min_o:.1f} minutes ({100 * (min_c - min_o) / min_o:.0f}% longer)")
print("Detour follows:", ' -> '.join(street_list(route_2011)))

# TODO 4: Map — grey network, rivers, closed bridges in red, both routes.
#   Three .plot() calls you have seen: edges (grey, thin), rivers (blue), then the
#   closed bridge edges in red with linewidth=6, then route_open and route_2011.
fig, ax = plt.subplots(figsize=(12, 9))
...
ax.legend(loc='lower right')
ax.set_title('Detour caused by the February 2011 Avon bridge closures', fontsize=13)
ax.set_xlabel('Easting (m)'); ax.set_ylabel('Northing (m)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

### Which bridge matters most?
Engineers rank assets by **criticality**: the consequence of losing them. With a routing
function that is a loop — close each Avon bridge on its own and measure the delay it causes.

In [ ]:
# Criticality of each eastern Avon bridge for the New Brighton Police Station -> hospital trip
results = []
for bridge in closed_2011:
    open_b = edges[~(edges['river'].notna() & (edges['bridge_name'] == bridge))]
    graph_b = build_graph(open_b, 'travel_s')
    dist_b, _ = dijkstra(graph_b, origin_p, destination_p)
    results.append({'bridge closed': bridge,
                    'trip time (min)': dist_b[destination_p] / 60,
                    'delay (min)': dist_b[destination_p] / 60 - min_o})

criticality = pd.DataFrame(results).sort_values('delay (min)', ascending=False)
print(f"Baseline, all bridges open: {min_o:.1f} min\n")
print(criticality.to_string(index=False, float_format=lambda x: f'{x:.1f}'))

> **Redundancy.** Only one bridge shows a delay on its own — the fastest route simply does not
> use the others, and when Pages Road closes there is another crossing a few blocks away. That
> is network redundancy, and it is why the *combined* 2011 closure hurt far more than any single
> bridge: resilience planning has to look at simultaneous failures, not one asset at a time.

## 7. Service Area: How Far Is the Hospital?

Call `dijkstra()` **without a target** and it settles every node — one run gives the travel
time from Christchurch Hospital to *all* 6,946 intersections. Colouring the nodes by that time
is an **isochrone map**, the standard tool for asking "who is within 10 minutes of the
hospital?" or "where should the next fire station go?".

Cartography check (Lecture 7): travel time is a quantity, so use a **sequential** colour ramp
with a small number of classes and a legend that states the units.

In [ ]:
# Travel time from the hospital to every node, with all bridges open and with the 2011 closures
hospital = nearest_node(place('Christchurch Hospital'))

time_open, _   = dijkstra(time_graph, hospital)        # no target -> settle every node
time_closed, _ = dijkstra(graph_2011, hospital)

nodes['min_open']   = nodes['node_id'].map(time_open) / 60
nodes['min_closed'] = nodes['node_id'].map(time_closed) / 60
nodes['extra_min']  = nodes['min_closed'] - nodes['min_open']

print(f"Nodes within 10 min of the hospital (all open):    {(nodes['min_open'] <= 10).sum():>5}")
print(f"Nodes within 10 min of the hospital (2011 closures):{(nodes['min_closed'] <= 10).sum():>5}")
print(f"Nodes delayed by more than 2 min by the closures:   {(nodes['extra_min'] > 2).sum():>5}")

In [ ]:
# Isochrone map and the delay map, side by side
import matplotlib.colors as mcolors

bins   = [0, 5, 10, 15, 20, 40]
labels = ['0-5', '5-10', '10-15', '15-20', '20+']
nodes['class_open'] = pd.cut(nodes['min_open'], bins=bins, labels=labels, include_lowest=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

ax = axes[0]
edges.plot(ax=ax, color='#e1e6e8', linewidth=0.4, zorder=1)
nodes.plot(ax=ax, column='class_open', cmap='YlOrRd', markersize=6, legend=True, zorder=2,
           legend_kwds={'title': 'Minutes from hospital', 'loc': 'lower right'})
rivers.plot(ax=ax, color='#2980b9', linewidth=1.5, zorder=3)
ax.scatter(place('Christchurch Hospital').x, place('Christchurch Hospital').y,
           s=250, marker='P', color='black', zorder=4)
ax.set_title('Drive time from Christchurch Hospital (all bridges open)', fontsize=12)
ax.set_aspect('equal'); ax.set_axis_off()

ax = axes[1]
edges.plot(ax=ax, color='#e1e6e8', linewidth=0.4, zorder=1)
affected = nodes[nodes['extra_min'] > 0.1]
affected.plot(ax=ax, column='extra_min', cmap='PuRd', markersize=8, legend=True, zorder=2,
              vmin=0, vmax=5, legend_kwds={'label': 'Extra minutes', 'shrink': 0.6})
edges[edges['river'].notna() & edges['bridge_name'].isin(closed_2011)].plot(
    ax=ax, color='red', linewidth=5, zorder=4)
rivers.plot(ax=ax, color='#2980b9', linewidth=1.5, zorder=3)
ax.set_title('Extra drive time caused by the 2011 bridge closures', fontsize=12)
ax.set_aspect('equal'); ax.set_axis_off()

plt.tight_layout()
plt.show()

## 8. Summary and Reflection

### What You Learned Today
1. **Graphs**: a road network is nodes (intersections), edges (segments) and weights (cost) — connectivity, not geometry, is what routing uses
2. **Adjacency dictionary**: `graph[u][v] = weight` is all the data structure a routing engine needs
3. **Dijkstra's algorithm**: settle the closest unvisited node, relax its edges, repeat — you wrote it from pseudocode and verified it against a hand-worked example
4. **Snapping**: real places are joined to the network through the nearest node
5. **Weights matter**: the shortest route and the fastest route are usually different roads
6. **Disruption analysis**: remove edges, rebuild the graph, reroute — the consequence of a closure is measured in minutes, and looping over assets ranks their criticality
7. **Service areas**: one Dijkstra run with no target gives the travel time to every node — the basis of isochrone maps and facility-location studies

### Engineering Takeaways
- **Every navigation app** is Dijkstra (or a close relative such as A*) on a graph like the one you built, with live traffic replacing the speed limit as the weight
- **Free-flow time is an estimate**: our `travel_s` ignores traffic, turns and signals — a model is only as good as its weights
- **Topology beats geometry**: two lines crossing on a map are not connected unless they share a node. Data from OpenStreetMap or LINZ carries that topology; a plain drawing does not
- **Bridges are critical links**: the 2011 closures did not isolate anyone, but they added minutes to every eastern trip — exactly the kind of quantified consequence a resilience plan needs

### Reflection Questions
1. We treated every street as two-way. Which of your routes would change if one-way streets in the central city were respected, and how would you represent one-way streets in the adjacency dictionary?
2. Dijkstra settles the *closest* node first. Why does that guarantee its distance is final? What would go wrong if an edge had a negative weight?
3. Emergency vehicles can exceed the speed limit but still cannot cross a closed bridge. Which of the two effects — speed or connectivity — did the 2011 closures change, and why does that make bridges more critical than slow roads?

---

### Assessment: Lab Sign-Off

**This lab is not complete until you have submitted your sign-off to LEARN.**

**Step 1 — Self-check.** Make sure you can do each of these:

- [ ] Built distance- and time-weighted graphs from the `edges` layer
- [ ] Implemented `dijkstra()` and passed the toy-network tests
- [ ] Routed between two real places and explained why the fastest and shortest routes differ
- [ ] Modelled the 2011 bridge closures and reported the extra travel time
- [ ] Interpreted the service-area (isochrone) map

**Step 2 — Take the sign-off quiz.**

1. Download **`Lab4_SignOff.html`** from LEARN and open it in your web browser.
2. Enter your full name and University ID, then answer the 10 questions.
3. You need **8/10** to pass. You can retake it as often as you like — each
   attempt draws a new set of questions, and every answer comes with an explanation.
4. When you pass, click **Download Sign-Off PNG**. This saves a file called
   `ENCN205_Lab4_<yourID>.png` to your computer.

**Step 3 — Submit.** Upload `ENCN205_Lab4_<yourID>.png` to the **Lab 4 Sign-Off**
submission on LEARN.

> Your PNG records your name, ID, score, and a timestamp, encoded in the QR code.
> It is your proof of completion — show it to your tutor if they ask.